In [ ]:
import __main__
import sys, os
project_root = os.path.abspath("..")  # adjust if notebook is elsewhere
sys.path.insert(0, project_root)
from typing import Dict, List, Literal, Tuple, Optional, Any
import logging

import category_encoders as ce
import matplotlib.pyplot as plt

import numexpr as ne # makes numpy operations faster
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

from catboost import CatBoostRegressor, CatBoostClassifier

from geo_functions import compute_cyclicity_score, split_dataset_to_linear_and_cyclic, make_windows_from_data

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    # print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

import src.param_config.config_paths as P

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
"another weather (https://www.kaggle.com/datasets/muthuj7/weather-dataset?select=weatherHistory.csv)"

df = pd.read_csv('../public_datasets/2D/tabular/another_weather/weatherHistory.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")


In [ ]:
"[works] china weather"

file_location = '../public_datasets/3D/china_weather/china_weather_000.npy'
weather_array = np.load(file_location).transpose(0, 2, 1)

for col in range(weather_array.shape[2]):
    data_col    = weather_array[2, :, col].flatten()
    cycle_score = compute_cyclicity_score(data_col)
    print(f"Cyclicity score of property {col}: {cycle_score}")

y_indices       = [7, 10] #[4, 5, 6, 7, 10]

mask            = np.ones(weather_array.shape[2], dtype=bool)
mask[y_indices] = False
X_cut           = weather_array[:, :, mask]      # (stations, timesteps, n_features)
y_cut           = weather_array[:, :, y_indices] # (stations, timesteps, n_targets)

assert X_cut.shape[2] + y_cut.shape[2] == weather_array.shape[2]

page_choice = 7
X = pd.DataFrame(X_cut[page_choice])
y = y_cut[page_choice]

print(f"X shape: {X.shape}, y shape: {y.shape}")


In [ ]:
"[works] longterm weather (https://www.kaggle.com/datasets/alistairking/weather-long-term-time-series-forecasting)"

df = pd.read_csv('../public_datasets/2D/tabular/longterm_weather/longterm_weather.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")

y_cols = ["rain"]
y      = df[y_cols].to_numpy()
X      = df.drop(columns=y_cols, inplace=False)


In [ ]:
"cities weather (https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data?resource=download)"

weather_folder   = "../public_datasets/2D/tabular/Hourly Weather Data 2012-2017"
files_to_combine = ["temperature.csv", "humidity.csv", "wind_speed.csv", "pressure.csv", "wind_direction.csv"]

dfs = {f.split(".")[0]: pd.read_csv(os.path.join(weather_folder, f)) for f in files_to_combine}

# Extract city names (assumes all files have same columns)
cities     = [col for col in dfs["temperature"].columns if col != "datetime"]
timesteps  = len(dfs["temperature"])
properties = len(files_to_combine)

weather_array = np.zeros((len(cities), timesteps, properties), dtype=float)

# Fill array
for p, prop in enumerate(files_to_combine):
    df_prop = dfs[prop.split(".")[0]]  # remove .csv from name
    for c, city in enumerate(cities):
        weather_array[c, :, p] = df_prop[city].values

for c in range(weather_array.shape[0]):        # cities
    for p in range(weather_array.shape[2]):    # properties
        col = weather_array[c, :, p]
        if np.isnan(col).any():
            mean_val = np.nanmean(col)  # compute mean ignoring NaNs
            col[np.isnan(col)] = mean_val
            weather_array[c, :, p] = col
print("Array shape:", weather_array.shape)
print("NaNs remaining:", np.isnan(weather_array).sum())

cycle_score = compute_cyclicity_score(weather_array[0, :, 0])
print(f"Cyclicity score : {cycle_score}")



In [ ]:
"Pseudo-cyclic synthetic dataset (https://archive.ics.uci.edu/dataset/136/pseudo+periodic+synthetic+time+series)"

data = np.loadtxt('../public_datasets/2D/tabular/pseudo_cyclic/synthetic.data')

for i in range(data.shape[1]):
    cycle_score = compute_cyclicity_score(data[:,i])
    print(f"Cyclicity score of feature {i}: {cycle_score}")

In [ ]:
"Traffic flow dataset (https://archive.ics.uci.edu/dataset/608/traffic+flow+forecasting)"

from scipy.io import loadmat

# Load data
data = loadmat("../public_datasets/2D/tabular/traffic_dataset/traffic_dataset.mat")

def flatten_X(mat_array):
    """
    Convert 1xN MATLAB object array of 36x48 matrices into 2D numeric array
    N rows, 36*48 columns
    """
    flattened = []
    for m in mat_array[0]:
        # Ensure numeric type
        flattened.append(np.array(m, dtype=float).reshape(-1))
    return np.stack(flattened, axis=0)

# Flatten input features
X_train_np = flatten_X(data['tra_X_tr'])
X_test_np  = flatten_X(data['tra_X_te'])

# Outputs: already numeric, just transpose to N x 36
y_train_np = data['tra_Y_tr'].T.astype(float)
y_test_np  = data['tra_Y_te'].T.astype(float)

# Convert to Polars
X_train = pl.DataFrame(X_train_np)
X_test  = pl.DataFrame(X_test_np)
y_train = pl.DataFrame(y_train_np)
y_test  = pl.DataFrame(y_test_np)

print(X_train.shape, y_train.shape)
print(X_train.head())
print(y_train.head())



In [ ]:
"Electric power data (https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption)"

from ucimlrepo import fetch_ucirepo

def get_household_power_consumption(target: str = "Global_active_power") -> Tuple[pl.DataFrame, pl.Series]:
    # 1. Fetch dataset
    ds = fetch_ucirepo(id=235)
    
    # 2. Combine and clean in Pandas to handle mixed types ('?')
    # For this dataset, ds.data.original contains all columns combined
    df_pd = ds.data.original.copy()
    
    # Coerce all columns to numeric except Date and Time
    for col in df_pd.columns:
        if col not in ["Date", "Time"]:
            df_pd[col] = pd.to_numeric(df_pd[col], errors='coerce')
            
    # 3. Convert to Polars safely
    df = pl.from_pandas(df_pd)
    
    # 4. Handle target and nulls
    if target not in df.columns:
        raise ValueError(f"Target '{target}' not found")
        
    df = df.filter(pl.col(target).is_not_null())
    y = df.get_column(target)
    X = df.drop(target).fill_null(0)
    return X, y

X, y = get_household_power_consumption()
print(f"X shape: {X.shape}, y shape: {y.shape}")


numeric_types = [pl.Float32, pl.Float64, pl.Int32, pl.Int64]

for col in X.columns:
    if X[col].dtype not in numeric_types:
        continue
    cycle_score = compute_cyclicity_score(X[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")



In [ ]:
# utils functions

def reparam_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """Gaussian reparameterization."""
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

def reparam_vmf(mu_dir: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
    """vMF has no closed form, so approximate vMF sampling: Gaussian noise + projection. Radius = 1 by construction.
    This is what Davidson (2018) does as well, fine because ELBO needs approximate sampling"""
    eps = torch.randn_like(mu_dir)
    z   = mu_dir + eps / (kappa + 1e-6)
    return F.normalize(z, dim= -1) #this projects to unit sphere

class EuclidEncoder(nn.Module):
    def __init__(self, window_size, input_dim, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * input_dim, hidden),
            nn.ReLU())
        self.mu     = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)

    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

class SphericalEncoder(nn.Module):
    def __init__(self, window_size, input_dim, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * input_dim, hidden),
            nn.ReLU())
        self.mu_raw = nn.Linear(hidden, z_dim)
        self.kappa  = nn.Linear(hidden, 1)

    def forward(self, x):
        h      = self.net(x)
        mu_dir = F.normalize(self.mu_raw(h), dim=-1)
        kappa  = F.softplus(self.kappa(h)) + 1e-3
        return mu_dir, kappa

class Decoder(nn.Module):
    def __init__(self, z_dim_total, window_size, output_dim, hidden):
        super().__init__()
        self.window_size = window_size
        self.output_dim   = output_dim  # number of features (linear+cyclic)
        self.net = nn.Sequential(
            nn.Linear(z_dim_total, hidden),
            nn.ReLU(),
            nn.Linear(hidden, window_size * output_dim))  # << flattened output

    def forward(self, z):
        # z: (B, z_dim_total)
        return self.net(z)  # shape (B, window_size*output_dim)


def kl_gaussian(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu**2 - logvar.exp(), dim=1).mean()

def regularization_vmf(kappa, dim):
    """Spherical has no closed form of the KL term, so this functino is a proxy regularizer
    This is more of a concentration regularizer encouraging proximity to a uniform hyperspherical prior than a KL divergence."""
    return (kappa - (dim - 1) * torch.log(kappa + 1e-6)).mean()


# with split
@torch.no_grad()
def encode_dataset(lin_windows, cyc_windows, lin_encoder, cyc_encoder, pooling="mean"):
    """Encode linear + cyclic windows.
    lin_windows: (num_windows, win, D_lin)
    cyc_windows: (num_windows, win, D_cyc)"""
    N, win, D_lin = lin_windows.shape
    D_cyc         = cyc_windows.shape[-1]

    lin_flat = lin_windows.view(N, win*D_lin)
    cyc_flat = cyc_windows.view(N, win*D_cyc)

    mu_e, logvar_e = lin_encoder(lin_flat)
    mu_s, kappa    = cyc_encoder(cyc_flat)

    z = torch.cat([mu_e, mu_s], dim=-1)

    if pooling is None:
        return z.view(N, 1, -1)  # add dummy W dim
    if pooling == "mean":
        return z.mean(dim=0, keepdim=True)  # only one sequence
    if pooling == "max":
        return z.max(dim=0, keepdim=True).values
    raise ValueError(f"Unknown pooling: {pooling}")

def vae_train_step(x_lin: torch.Tensor, x_cyc: torch.Tensor, lin_encoder: nn.Module, cyc_encoder: nn.Module,
                   decoder: nn.Module, lambdas: dict) -> torch.Tensor:
    """Single VAE training step.
    - x_lin, x_cyc : (batch, window_size, features)"""
    B, win, D_lin = x_lin.shape
    D_cyc         = x_cyc.shape[-1]

    x_lin_flat = x_lin.view(B, win*D_lin)
    x_cyc_flat = x_cyc.view(B, win*D_cyc)

    mu_e, logvar_e = lin_encoder(x_lin_flat)
    mu_s, kappa    = cyc_encoder(x_cyc_flat)

    z_e = reparam_gaussian(mu_e, logvar_e)
    z_s = reparam_vmf(mu_s, kappa)
    z   = torch.cat([z_e, z_s], dim=-1)

    x_hat       = decoder(z)
    x_full_flat = torch.cat([x_lin_flat, x_cyc_flat], dim=-1)
    L_recon  = F.mse_loss(x_hat, x_full_flat)
    L_kl_e   = kl_gaussian(mu_e, logvar_e)
    L_reg_s  = regularization_vmf(kappa, z_s.size(-1))

    return lambdas["rec"]*L_recon + lambdas["euc"]*L_kl_e + lambdas["sph"]*L_reg_s

def train_vae(loader: DataLoader, lin_encoder: nn.Module, cyc_encoder: nn.Module,
              decoder: nn.Module, optimizer: torch.optim.Optimizer, lambdas: dict) -> None:
    """Full training loop over one epoch for the VAE.
    Args:
        loader      : DataLoader yielding (x_lin, x_cyc)
        lin_encoder : Linear encoder
        cyc_encoder : Cyclic encoder
        decoder     : Decoder
        optimizer   : Optimizer
        lambdas     : dict of loss weights"""
    lin_encoder.train()
    cyc_encoder.train()
    decoder.train()

    for x_lin, x_cyc in loader:
        optimizer.zero_grad()
        loss = vae_train_step(x_lin, x_cyc, lin_encoder, cyc_encoder, decoder, lambdas)
        loss.backward()
        optimizer.step()
        print(f"Loss: {loss.item():.4f}")


# no split
def vae_train_step_no_split(x: torch.Tensor, encoder: nn.Module, decoder: nn.Module, lambdas: dict) -> torch.Tensor:
    """Single VAE step for dataset without linear/cyclic split.
    x : (num_windows, window_size, num_features)"""
    N_w, win, D = x.shape
    x_flat      = x.view(N_w, win*D)

    mu, logvar  = encoder(x_flat)
    z           = reparam_gaussian(mu, logvar)

    x_hat   = decoder(z)
    L_recon = F.mse_loss(x_hat, x_flat)
    L_kl    = kl_gaussian(mu, logvar)

    return lambdas["rec"]*L_recon + lambdas["euc"]*L_kl

def train_vae_no_split(loader, encoder, decoder, optimizer, lambdas):
    encoder.train(); decoder.train()
    for x, in loader:
        optimizer.zero_grad()
        loss = vae_train_step_no_split(x, encoder, decoder, lambdas)
        loss.backward()
        optimizer.step()
        print(f"Loss: {loss.item():.4f}")

@torch.no_grad()
def encode_dataset_no_split(x: torch.Tensor, encoder: nn.Module, pooling: str | None = "mean") -> torch.Tensor:
    """Encode a dataset without linear/cyclic split.
    x : (num_windows, window_size, num_features)
    pooling : "mean" or None
    Returns : (num_windows, z_dim) if pooled, else (num_windows, window_size, z_dim)"""
    N_w, win, D = x.shape
    x_flat      = x.view(N_w, win*D)
    mu, _       = encoder(x_flat)
    
    if pooling is None:
        return mu
    if pooling == "mean":
        return mu.mean(dim=0, keepdim=True)  # average over windows
    raise ValueError(f"Unknown pooling: {pooling}")


def fit_catboost_multi(X_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray,) -> np.ndarray:
    """Fit CatBoost for multi-output regression."""
    model = MultiOutputRegressor(CatBoostRegressor(verbose=0))
    model.fit(X_train, y_train)
    return model.predict(X_test)



In [ ]:
"Params"
window_size = 8
epochs      = 30
z_dim_total = 16
hidden_dim  = 128
lr_optimizer= 1e-3
batch_size  = 128

# for split scenario
z_dim_euclid  = z_dim_total // 2
z_dim_spheric = z_dim_total - z_dim_euclid
cyclic_threshold = 0.1


"Shared preprocessing steps"
dataset      = "china_weather"
cols_to_drop = {"longterm_weather": ["date"],
                "electric_power":   ["Date", "Time"],
                "china_weather":    [],}
X.drop(columns=cols_to_drop[dataset], inplace=True, errors='ignore') # drop non-numeric if present

# 1. train/test split without shuffling (time series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False) #timeseries order matters


In [ ]:
"DIRECT PREDICTION"

# 2. scale X and y
X_scaler = StandardScaler()
X_train  = X_scaler.fit_transform(X_train)
X_test   = X_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train  = y_scaler.fit_transform(y_train).ravel()
y_test   = y_scaler.transform(y_test).ravel()

# 3. CatBoost (direct regression) predict + metrics
y_hat = fit_catboost_multi(X_train, y_train, X_test)
rmse  = np.sqrt(mean_squared_error(y_test, y_hat))
r2    = r2_score(y_test, y_hat)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
"NO SPLIT"

# 2. scale X and y
y_scaler  = StandardScaler()
y_train   = y_scaler.fit_transform(y_train)
y_test    = y_scaler.transform(y_test)

X_scaler  = StandardScaler()
X_train   = pd.DataFrame(X_scaler.fit_transform(X_train), columns=X_train.columns)
X_test    = pd.DataFrame(X_scaler.transform(X_test), columns=X_train.columns)  # use train cols

# 3. sliding windows
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
X_test_t  = torch.tensor(X_test.values,  dtype=torch.float32)
X_train_w = make_windows_from_data(X_train_t, window_size).to(device)
X_test_w  = make_windows_from_data(X_test_t,  window_size).to(device)

# 4. encoder & decoder (only Euclidean encoder)
encoder   = EuclidEncoder(window_size=window_size, input_dim=X_train.shape[1],
                          z_dim=z_dim_total, hidden=hidden_dim).to(device)
decoder   = Decoder(z_dim_total=z_dim_total, window_size=window_size, output_dim=X_train.shape[1], hidden=hidden_dim).to(device)
optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=lr_optimizer)

# 5. dataset and loader
train_ds     = TensorDataset(X_train_w)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# 6. lambdas: no spherical component + train
lambdas = {"rec": 1.0, "euc": 1e-3, "sph": 0.0}

for epoch in range(epochs):
    train_vae_no_split(train_loader, encoder, decoder, optimizer, lambdas)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}")

# 9. encode
Z_train = encode_dataset_no_split(X_train_w, encoder, pooling=None)
Z_test  = encode_dataset_no_split(X_test_w, encoder, pooling=None)

# 10. align y to number of windows
y_train_win = y_train[:Z_train.shape[0]]
y_test_win  = y_test[:Z_test.shape[0]]

# 11. CatBoost
# model = CatBoostRegressor(verbose=0)
# model.fit(Z_train.cpu().numpy(), y_train_win)
# y_hat = model.predict(Z_test.cpu().numpy())
y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())

rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
r2    = r2_score(y_test_win, y_hat)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")



In [ ]:
"CYCLIC + LINEAR split"

# 2. should split to linear/cyclic BEFORE scaling, as scaling changes the cyclicity measure
X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=cyclic_threshold)

# select same cols as X_train
X_lin_test   = X_test[X_lin_train.columns]
X_cyc_test   = X_test[X_cyc_train.columns]

# 3. scale linear and cyclic SEPARATELY
y_scaler     = StandardScaler()
y_train      = y_scaler.fit_transform(y_train)
y_test       = y_scaler.transform(y_test)

X_lin_scaler = StandardScaler()
X_lin_train  = pd.DataFrame(X_lin_scaler.fit_transform(X_lin_train), columns=X_lin_train.columns)
X_lin_test   = pd.DataFrame(X_lin_scaler.transform(X_lin_test), columns=X_lin_train.columns)  # use train columns

X_cyc_scaler = StandardScaler()
X_cyc_train  = pd.DataFrame(X_cyc_scaler.fit_transform(X_cyc_train), columns=X_cyc_train.columns)
X_cyc_test   = pd.DataFrame(X_cyc_scaler.transform(X_cyc_test), columns=X_cyc_train.columns)  # use train columns

# convert to torch tensors AFTER scaling
X_lin_train  = torch.tensor(X_lin_train.values, dtype=torch.float32)
X_lin_test   = torch.tensor(X_lin_test.values, dtype=torch.float32)
X_cyc_train  = torch.tensor(X_cyc_train.values, dtype=torch.float32)
X_cyc_test   = torch.tensor(X_cyc_test.values, dtype=torch.float32)

# window data. we do it after splitting to linear/cyclic, as the sliding window would mess the cyclicity measure if done before
X_lin_train_w = make_windows_from_data(X_lin_train, window_size).to(device)
X_cyc_train_w = make_windows_from_data(X_cyc_train, window_size).to(device)
X_lin_test_w  = make_windows_from_data(X_lin_test, window_size).to(device)
X_cyc_test_w  = make_windows_from_data(X_cyc_test, window_size).to(device)
X_train_w     = torch.cat([X_lin_train_w, X_cyc_train_w], dim=-1).to(device)
X_test_w      = torch.cat([X_lin_test_w, X_cyc_test_w], dim=-1).to(device)

encoder_e = EuclidEncoder(window_size=window_size, input_dim=X_lin_train.shape[1], z_dim=z_dim_euclid, hidden=hidden_dim).to(device)
encoder_s = SphericalEncoder(window_size=window_size, input_dim=X_cyc_train.shape[1], z_dim=z_dim_spheric, hidden=hidden_dim).to(device)
decoder   = Decoder(z_dim_total=z_dim_euclid + z_dim_spheric, window_size=window_size, output_dim=X_train.shape[1], hidden=hidden_dim).to(device)
optimizer = torch.optim.AdamW(list(encoder_e.parameters()) + list(encoder_s.parameters())+ list(decoder.parameters()), lr=lr_optimizer)

# training
train_ds     = TensorDataset(X_lin_train_w, X_cyc_train_w)#, X_train_w)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

lambdas = {"rec": 1.0, "euc": 1e-3, "sph": 1e-3}

for epoch in range(epochs):
    train_vae(train_loader, encoder_e, encoder_s, decoder, optimizer, lambdas)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}")

# encode without collapsing all windows
Z_train = encode_dataset(X_lin_train_w, X_cyc_train_w, encoder_e, encoder_s, pooling=None)
Z_train = Z_train.mean(dim=1)  # mean over windows of each sample
Z_test  = encode_dataset(X_lin_test_w, X_cyc_test_w, encoder_e, encoder_s, pooling=None)
Z_test  = Z_test.mean(dim=1)

# Cut y_train/y_test to match number of windows
y_train_win = y_train[:Z_train.shape[0]]
y_test_win  = y_test[:Z_test.shape[0]]

# model = CatBoostRegressor(verbose=0)
# model.fit(Z_train.cpu().numpy(), y_train_win)
# y_hat = model.predict(Z_test.cpu().numpy())
y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())
rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
r2    = r2_score(y_test_win, y_hat)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
# dimensionality estimation on z
import geomstats.geometry.hypersphere as hs
from geomstats.learning.pca import TangentPCA

latent_dim = 20
vae_model = train_vae(X_train, latent_dim=latent_dim, epochs=100, lr=1e-3,
                      vae_layer_dims=layer_dims, beta=beta_vae)
with torch.no_grad():
    mu_train, logvar_train = vae_model.encode(X_train)
    eps_train   = torch.randn_like(mu_train)
    z_train_vae = mu_train + eps_train * torch.exp(0.5 * logvar_train)
    z_train_vae = z_train_vae.cpu().numpy()

pca = PCA()
pca.fit(z_train_vae)
explained     = np.cumsum(pca.explained_variance_ratio_)
intrinsic_dim = np.searchsorted(explained, 0.9) + 1  # 90% variance
print("Estimated intrinsic dim (PCA):", intrinsic_dim)

def twoNN(X):
    nbrs   = NearestNeighbors(n_neighbors=3).fit(X)
    distances, _ = nbrs.kneighbors(X)
    r2     = distances[:,2] / distances[:,1]  # 2nd NN / 1st NN
    id_est = (np.mean(np.log(r2)))**(-1)
    return id_est

print("Estimated intrinsic dim (TwoNN):", twoNN(z_train_vae))

sphere = hs.Hypersphere(dim=latent_dim)  # e.g., S^k
z_proj = z_train_vae / np.linalg.norm(z_train_vae, axis=1, keepdims=True)  # project onto sphere
pga    = TangentPCA(sphere, n_components=None)
pga.fit(z_proj)
explained     = np.cumsum(pga.explained_variance_ratio_)
intrinsic_dim = np.searchsorted(explained, 0.9) + 1
print("Estimated intrinsic dim (PGA):", intrinsic_dim)


In [ ]:
"kernel PCA"
from sklearn.decomposition import KernelPCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

# ===== 1. Sweep embedding dimension to estimate residual variance (optional) =====
# Note: KernelPCA does not give a built-in dist_matrix_, so residual variance calculation is less straightforward.
# Here we can skip it or just look at explained variance if using linear kernel.
# For nonlinear kernels, you typically pick n_components based on domain knowledge or grid search.

# ===== 2. Fit kernel PCA with chosen embedding dimension =====
n_components = 8  # same as your Isomap example
kpca_model   = KernelPCA(n_components=n_components, kernel='rbf', gamma=0.05, fit_inverse_transform=True)
X_train_kpca = kpca_model.fit_transform(X_train)
X_test_kpca  = kpca_model.transform(X_test)

# ===== 3. Plot first 3 components (for visualization) =====
fig = plt.figure(figsize=(8,6))
ax  = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_kpca[:,0],
    X_train_kpca[:,1],
    X_train_kpca[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')
ax.set_title("Kernel PCA 3D Embedding")
plt.show()

# ===== 4. RandomForestClassifier on KPCA embedding =====
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_kpca, y_train_np)
y_pred_class = rf_classifier.predict(X_test_kpca)
accuracy     = accuracy_score(y_test_np, y_pred_class)
rmse_class   = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Kernel PCA accuracy (classif.): {accuracy:.3f}")
print(f"Kernel PCA RMSE (classif.): {rmse_class:.3f}")

# ===== 5. RandomForestRegressor on KPCA embedding =====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_kpca, y_train_np)
y_pred_reg = rf_regressor.predict(X_test_kpca)
r2        = r2_score(y_test_np, y_pred_reg)
rmse_reg  = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Kernel PCA R² (regression): {r2:.3f}")
print(f"Kernel PCA RMSE (regression): {rmse_reg:.3f}")


In [ ]:
"isomap"
from sklearn.manifold import Isomap
import warnings
from scipy.sparse import SparseEfficiencyWarning
warnings.simplefilter('ignore', SparseEfficiencyWarning)

# ===== 1. plot residual variance to estimate intrinsic dimensionality (finds the best dim)
res_vars       = []
embedding_dims = range(1, 11)  # try 1D to 10D embeddings
for dim in embedding_dims:
    iso = Isomap(n_neighbors=20, n_components=dim)
    iso.fit(X_train)
    # residual variance = 1 - R^2 between graph distances and embedding distances
    dist_graph = iso.dist_matrix_
    dist_emb   = np.linalg.norm(iso.embedding_[:, None, :] - iso.embedding_[None, :, :], axis=2)
    r2         = np.corrcoef(dist_graph.ravel(), dist_emb.ravel())[0,1]**2
    res_vars.append(1 - r2)
plt.plot(embedding_dims, res_vars, marker='o')
plt.xlabel("Embedding dimension")
plt.ylabel("Residual variance")
plt.title("Estimate intrinsic dimensionality")
plt.show()

# ===== 2. plot isomap of training data (uses the best estimated dim from above)
isomap_model   = Isomap(n_neighbors=20, n_components=5)
X_train_isomap = isomap_model.fit_transform(X_train)
X_test_isomap  = isomap_model.transform(X_test)

fig     = plt.figure(figsize=(8,6))
ax      = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_isomap[:,0],
    X_train_isomap[:,1],
    X_train_isomap[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')  # attach to figure
ax.set_title("Isomap 3D Embedding")
plt.show()


In [ ]:
"isomap results"
rf_classifier= RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_isomap, y_train_np)          # train on Isomap embedding
y_pred_class = rf_classifier.predict(X_test_isomap)          # predict test labels
accuracy     = accuracy_score(y_test_np, y_pred_class)
print(f"Isomap accuracy (classif.): {accuracy:.3f}")
rmse         = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Isomap RMSE (classif.): {rmse:.3f}")

# ====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_isomap, y_train_np)           # train on Isomap embedding
y_pred_reg   = rf_regressor.predict(X_test_isomap)      # predict on test embedding
r2           = r2_score(y_test_np, y_pred_reg)
print(f"Isomap R² (regression): {r2:.3f}")
test_rmse    = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Isomap RMSE (regression): {test_rmse:.3f}")


In [ ]:
"new metrics"
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score

def propagate_knn(Z_latent, y_true, missing_frac=0.2, k=15, tau=1.0):
    """Perform kNN label propagation on latent vectors."""
    num_rows = Z_latent.shape[0]
    dist_matrix = np.zeros((num_rows, num_rows))
    
    # compute L2 distances
    for i in range(num_rows):
        u = Z_latent[i]
        for j in range(i, num_rows):
            v = Z_latent[j]
            d = np.linalg.norm(u - v)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # mask random fraction
    num_missing = int(missing_frac * num_rows)
    missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)
    y_input     = y_true.copy().astype(float)
    y_input[missing_idx] = np.nan

    # kNN neighbors
    neighbors_idx = np.argsort(dist_matrix, axis=1)[:, 1:k+1]

    # edge weights
    def compute_weights(dists):
        w = np.exp(-dists / tau)
        return w / w.sum()

    # propagate labels
    y_prop = y_input.copy()
    for i in range(num_rows):
        if np.isnan(y_input[i]):
            neigh     = neighbors_idx[i]
            neigh_y   = y_input[neigh]
            mask      = ~np.isnan(neigh_y)
            if np.sum(mask) == 0:
                continue
            neigh_y   = neigh_y[mask]
            neigh_d   = dist_matrix[i, neigh][mask]
            w         = compute_weights(neigh_d)
            y_prop[i] = np.sum(w * neigh_y)

    return y_prop, missing_idx

def evaluate_metrics(y_true, y_pred, missing_idx):
    """Compute RMSE, MAE, Accuracy, Spearman correlation, and R² for masked points."""
    y_true_masked = y_true[missing_idx]
    y_pred_masked = y_pred[missing_idx]
    mask_valid    = ~np.isnan(y_pred_masked)
    
    if mask_valid.sum() == 0:
        return {k: np.nan for k in ["rmse","mae","acc","spearman","r2"]}
    
    y_true_valid = y_true_masked[mask_valid]
    y_pred_valid = y_pred_masked[mask_valid]


    def safe_spearmanr(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y).correlation

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred_valid)),
        "mae": mean_absolute_error(y_true_valid, y_pred_valid),
        "acc": accuracy_score(y_true_valid, np.round(y_pred_valid).astype(int)),
        "spearman": safe_spearmanr(y_true_valid, y_pred_valid),
        "r2": r2_score(y_true_valid, y_pred_valid)
    }
    return metrics

def format_metrics(metrics: dict) -> dict:
    """Format all float metrics to 3 decimal places, keep nan as-is."""
    return {k: (f"{v:.3f}" if isinstance(v, float) and not np.isnan(v) else v)
            for k, v in metrics.items()}

# --- Hybrid AE ---
y_prop_hybrid, missing_idx = propagate_knn(z_hybrid_encoded_np, df.iloc[:,1].values)
metrics_hybrid = evaluate_metrics(df.iloc[:,1].values, y_prop_hybrid, missing_idx)
print("Hybrid metrics:", format_metrics(metrics_hybrid))

# --- Full Euclidean AE ---
y_prop_euc, missing_idx = propagate_knn(z_euclid_full_np, df.iloc[:,1].values)
metrics_euc = evaluate_metrics(df.iloc[:,1].values, y_prop_euc, missing_idx)
print("Full Euclidean metrics:", format_metrics(metrics_euc))

# --- Baseline (mean predictor) ---
y_true      = df.iloc[:,1].values
num_rows    = len(y_true)
num_missing = int(0.2 * num_rows)
np.random.seed(42)
missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)

y_baseline = y_true.copy().astype(float)
y_baseline[missing_idx] = np.nan
mean_label = np.nanmean(y_baseline)
y_baseline[np.isnan(y_baseline)] = mean_label

metrics_baseline = evaluate_metrics(y_true, y_baseline, missing_idx)
print("Baseline metrics:", format_metrics(metrics_baseline))
